In [0]:
%run "../libs/libs"

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col, to_date

In [0]:
order_df = spark.read.format("delta").table("dev.tracking_sl.order")
product_df = spark.read.format("delta").table("dev.tracking_gd.dimproduct")
dimorder_df = spark.read.format("delta").table("dev.tracking_gd.dimorder")

In [0]:
product = product_df.alias('product')
order = order_df.alias('order')
dimorder = dimorder_df.alias('dimorder')

tdetail_df = (
    order.join(
        product,
        (order.productcategory == product['category']) &
        (order.productsize == product['size']) &
        (order.productvalue == product['prouctvalue']) &
        (order.productweight == product['weight']),
        "inner"
    )  
    .join(
        dimorder,
        (order.orderno == dimorder['orderno']),
        "inner"
    )     
.select(
        col('datekey'),
        col('orderid').alias('order_id'),
        col('productid').alias('product_id'),
        col('noofitems'),
        col('estimatedcost'),
        col('deliverycost')
    )
    .withColumn('ingestion_time', current_timestamp())
)

In [0]:
tdetail_df.createOrReplaceTempView("temp_factorder")

In [0]:
%sql
merge into dev.tracking_gd.factorder tgt
USING ( 
  SELECT DISTINCT * 
  FROM temp_factorder 
) src 
on tgt.order_id = src.order_id
when not matched then
  insert (datekey, order_id, product_id,noofitems,estimatedcost,deliverycost, ingestion_time)
  Values(src.datekey,src.order_id,src.product_id,src.noofitems,src.estimatedcost,src.deliverycost,src.ingestion_time)

In [0]:
%sql
select * from dev.tracking_gd.factorder limit 100